<a href="https://colab.research.google.com/github/azharabbas1234/DeepLearning/blob/main/construction_Estimate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
from sklearn.metrics import r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import Callback
from joblib import dump, load
import joblib # Needed for saving the OneHotEncoder later
from tensorflow.keras.models import save_model

# --- CORE FEATURES to be used in model training ---
CORE_FEATURES = [
    'District', 'Settlement_Type', 'Area_sft', 'Land_Type',
    'Building_Type', 'Floor_Count', 'Design_Complexity', 'BBS_kg_per_sft'
]
CATEGORICAL_FEATURES = ['District', 'Settlement_Type', 'Land_Type', 'Building_Type']
NUMERICAL_FEATURES = ['Area_sft', 'Floor_Count', 'Design_Complexity', 'BBS_kg_per_sft']
# -----------------------------------------------------------------

# --- 1. Custom R2 Callback (Still needed to see accuracy during training) ---
class R2ScoreCallback(Callback):
    def __init__(self, validation_data):
        super().__init__()
        self.X_val, self.y_val = validation_data

    def on_epoch_end(self, epoch, logs=None):
        y_pred = self.model.predict(self.X_val, verbose=0)
        r2 = r2_score(self.y_val, y_pred)
        logs['val_r2'] = r2
        print(f" - val_r2: {r2:.4f}", end='')

# --- 2. Load Data and Select Core Features ---
df = pd.read_csv("/construction_data_augmented.csv")
X = df[CORE_FEATURES]
y = df['Total_Estimate_PKR']

# --- 3. Target (y) Scaling ---
y_scaler = MinMaxScaler()
y_scaled = y_scaler.fit_transform(y.values.reshape(-1, 1))
dump(y_scaler, 'y_scaler_manual.pkl')
print("Target scaler saved as y_scaler_manual.pkl")

# --- 4. Manual Preprocessing Setup and Training/Test Split ---

# Split data FIRST
X_train, X_test, y_train_scaled, y_test_scaled = train_test_split(
    X, y_scaled, test_size=0.2, random_state=42
)

# Initialize Scalers/Encoders
scaler = StandardScaler()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False) # sparse_output=False makes the output an array

# --- 5. Fit and Transform Training Data ---

# 5a. Numerical Scaling
X_train_num = X_train[NUMERICAL_FEATURES]
scaler.fit(X_train_num)
X_train_num_scaled = scaler.transform(X_train_num)
dump(scaler, 'scaler_manual.pkl') # Save the scaler

# 5b. Categorical Encoding
X_train_cat = X_train[CATEGORICAL_FEATURES]
encoder.fit(X_train_cat)
X_train_cat_encoded = encoder.transform(X_train_cat)
joblib.dump(encoder, 'encoder_manual.pkl') # Save the encoder

# 5c. Combine everything for training
X_train_processed = np.hstack([X_train_num_scaled, X_train_cat_encoded])

# --- 6. Transform Test Data (DO NOT CALL .fit() on test data) ---

# 6a. Numerical Scaling
X_test_num = X_test[NUMERICAL_FEATURES]
X_test_num_scaled = scaler.transform(X_test_num)

# 6b. Categorical Encoding
X_test_cat = X_test[CATEGORICAL_FEATURES]
X_test_cat_encoded = encoder.transform(X_test_cat)

# 6c. Combine everything for testing
X_test_processed = np.hstack([X_test_num_scaled, X_test_cat_encoded])

print("Data preparation complete. Input shape:", X_train_processed.shape[1])

# --- 7. Build ANN Model ---
input_shape = X_train_processed.shape[1]
model = Sequential([
    Dense(128, activation='relu', input_shape=(input_shape,)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')

# --- 8. Train Model ---
val_split_index = int(0.9 * len(X_train_processed))
X_train_for_fit = X_train_processed[:val_split_index]
y_train_for_fit = y_train_scaled[:val_split_index]
X_val_for_callback = X_train_processed[val_split_index:]
y_val_for_callback = y_train_scaled[val_split_index:]

r2_callback = R2ScoreCallback(validation_data=(X_val_for_callback, y_val_for_callback))

print("\nTraining ANN model with manual preprocessing...")
model.fit(
    X_train_for_fit,
    y_train_for_fit,
    epochs=50,
    batch_size=32,
    validation_data=(X_val_for_callback, y_val_for_callback),
    callbacks=[r2_callback],
    verbose=1
)

# --- 9. Final Evaluation and Save Model ---
loss = model.evaluate(X_test_processed, y_test_scaled, verbose=0)
y_pred_scaled = model.predict(X_test_processed, verbose=0)
r2 = r2_score(y_test_scaled, y_pred_scaled)

print(f"\n--- MANUAL Model Final Evaluation ---")
print(f"Test Loss (Scaled MSE): {loss:.6f}")
print(f"Test R-squared (Accuracy): {r2:.4f}")

save_model(model, 'ann_model_manual.h5')
print("Trained MANUAL ANN model saved as ann_model_manual.h5")

Target scaler saved as y_scaler_manual.pkl
Data preparation complete. Input shape: 23

Training ANN model with manual preprocessing...
Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0342 - val_loss: 0.0042 - val_r2: 0.9019
Epoch 2/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0039 - val_loss: 0.0033 - val_r2: 0.9225
Epoch 3/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0030 - val_loss: 0.0035 - val_r2: 0.9185
Epoch 4/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0026 - val_loss: 0.0026 - val_r2: 0.9390
Epoch 5/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0023 - val_loss: 0.0023 - val_r2: 0.9463
Epoch 6/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0023 - val_loss: 0.0021 - val_r2: 0.9518
Epoch 7/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0019 - val_loss: 0.0020 - val_r2: 0.9535
Epoch 8/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0018 - val_loss: 0.0019 - val_r2: 0.9551
Epoch 9/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0018 - val_loss: 0.0027 - val_r2: 0.9371
Epoch 10/50
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0017 - val_loss: 0.00


--- MANUAL Model Final Evaluation ---
Test Loss (Scaled MSE): 0.001430
Test R-squared (Accuracy): 0.9660
Trained MANUAL ANN model saved as ann_model_manual.h5


In [3]:
import gradio as gr
import pandas as pd
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.losses import MeanSquaredError
from joblib import load
import joblib
import os

# --- FEATURE DEFINITIONS (Must match the training script) ---
CATEGORICAL_FEATURES = ['District', 'Settlement_Type', 'Land_Type', 'Building_Type']
NUMERICAL_FEATURES = ['Area_sft', 'Floor_Count', 'Design_Complexity', 'BBS_kg_per_sft']
# -----------------------------------------------------------------

# --- 1. Load Model and Preprocessor Artifacts ---
# We check if all required files exist before attempting to load
required_files = ['y_scaler_manual.pkl', 'scaler_manual.pkl', 'encoder_manual.pkl', 'ann_model_manual.h5']
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print(f"FATAL ERROR: The following model files are missing: {', '.join(missing_files)}.")
    print("Please ensure you ran 'train_ann_manual.py' successfully first.")
    # In a real app, you would exit or raise an error here.
    # For this demonstration, we'll try to load, expecting failure if files are missing.

try:
    # Load the manual artifacts
    y_scaler = load('y_scaler_manual.pkl')
    scaler = load('scaler_manual.pkl') # Standard Scaler for numerical features
    encoder = joblib.load('encoder_manual.pkl') # OneHotEncoder for categorical features

    # Load the model
    custom_objects = {'mse': MeanSquaredError()}
    model = load_model('ann_model_manual.h5', custom_objects=custom_objects)

    print("All manual model artifacts loaded successfully.")
except Exception as e:
    print(f"An error occurred during model loading: {e}")
    exit()


# --- 2. Define Prediction Function ---
def predict_estimate(
    District, Settlement_Type, Area_sft, Land_Type, Building_Type,
    Floor_Count, Design_Complexity, BBS_kg_per_sft
):
    """
    Takes 8 simple user inputs, manually preprocesses them (scales/encodes),
    and predicts the total estimate cost using the ANN model.
    """

    # Create a DataFrame from the 8 simple inputs
    input_data = pd.DataFrame({
        'District': [District],
        'Settlement_Type': [Settlement_Type],
        'Area_sft': [Area_sft],
        'Land_Type': [Land_Type],
        'Building_Type': [Building_Type],
        'Floor_Count': [Floor_Count],
        'Design_Complexity': [Design_Complexity],
        'BBS_kg_per_sft': [BBS_kg_per_sft]
    })

    # --- Manual Preprocessing Steps (Mirroring the training script) ---

    # 2a. Separate features
    X_num = input_data[NUMERICAL_FEATURES]
    X_cat = input_data[CATEGORICAL_FEATURES]

    # 2b. Transform Numerical (Scale)
    # The scaler object was saved during training and is loaded above
    X_num_scaled = scaler.transform(X_num)

    # 2c. Transform Categorical (Encode)
    # The encoder object was saved during training and is loaded above
    X_cat_encoded = encoder.transform(X_cat)

    # 2d. Combine and prepare for prediction (Numerical first, then Categorical)
    input_processed = np.hstack([X_num_scaled, X_cat_encoded])

    # --- Prediction ---

    # Step 1: Make the prediction (output is scaled, 0 to 1)
    prediction_scaled = model.predict(input_processed, verbose=0)[0][0]

    # Step 2: Convert the scaled prediction back to the real PKR amount (Inverse Transform)
    prediction_pkr = y_scaler.inverse_transform([[prediction_scaled]])[0][0]

    # Handle potential negative prediction due to inverse transform if prediction is outside training bounds
    if prediction_pkr < 0:
        prediction_pkr = 0

    return f"Estimated Total Cost: PKR {prediction_pkr:,.2f}"

# --- 3. Setup Gradio Interface Components ---
DISTRICTS = ['Ghanche', 'Ghizer', 'Nagar', 'Astore', 'Diamer', 'Skardu', 'Shigar', 'Hunza', 'Gilgit', 'Roundu']
SETTLEMENT_TYPES = ['Rural', 'Sub-Rural', 'Urban']
LAND_TYPES = ['Commercial', 'Uncultivated', 'Cultivated']
BUILDING_TYPES = ['Commercial', 'Road', 'Residential']

# These 8 fields are what the user sees
inputs = [
    gr.Dropdown(DISTRICTS, label="1. District (Location)", value='Gilgit'),
    gr.Dropdown(SETTLEMENT_TYPES, label="2. Settlement Type (Urban/Rural)", value='Urban'),
    gr.Number(label="3. Area (sq ft)", value=2500, minimum=100, step=1),
    gr.Dropdown(LAND_TYPES, label="4. Land Type", value='Commercial'),
    gr.Dropdown(BUILDING_TYPES, label="5. Building Type", value='Residential'),
    gr.Number(label="6. Floor Count", value=3, minimum=1, step=1),
    gr.Number(label="7. Design Complexity (1=Simple, 5=Complex)", value=3, minimum=1, maximum=5, step=1),
    gr.Number(label="8. Steel/Rebar per sq ft (BBS kg/sft)", value=0, minimum=0, step=0.1)
]

# --- 4. Launch Gradio Interface ---
iface = gr.Interface(
    fn=predict_estimate,
    inputs=inputs,
    outputs=gr.Text(label="Predicted Total Estimate"),
    title="Simplified Construction Cost Predictor (Manual ANN)",
    description="This predictive tool uses an Artificial Neural Network trained only on 8 core structural and location features. Note: All underlying cost factors are fixed based on typical values from the training data.",
    allow_flagging="never"
)

iface.launch(inbrowser=True)

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


All manual model artifacts loaded successfully.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e3e010c846caf1074a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
